## **AI Agent for Data Quality Check**

- This AI agent performs comprehensive data quality checks on one or two datasets. It generates detailed reports covering missing values, duplicates, data types, and anomalies. When two datasets are provided, it compares them to identify differences in structure, data consistency, and key column uniqueness, helping ensure data integrity and readiness for analysis.

- **single_dataset_quality_report** : for single dataset case.
- **Basic Data Summary**:
 - Calculate and report the total number of rows and columns in the input dataframe.

- **Missing and Duplicate Values**:
 - Counts the number of missing (null) values per column and the total number of duplicate rows.

- **Data Types and Uniqueness**:
 - Records the data type of each column and counts the number of unique values per column.

- **Anomaly Detection (Negative Values)**:
 - Checks all numeric columns for negative values and reports how many negative values appear in each.

- **Key Columns Duplicate Check**:
 - If key columns are specified, counts duplicates based only on those columns and calculates the percentage of such duplicates relative to the dataset size.

- **Uniqueness in Key Columns**:
For each key column, indicates whether the values in that column are unique, or flags if the column is missing from the dataset.

This function essentially provides a comprehensive data quality snapshot for a single dataset, including data structure, missingness, duplicates, anomalies, and uniqueness checks.

In [ ]:
import pandas as pd
import numpy as np

# function single_dataset_quality_report takes two parameters
 #(pandas dataframe and key_columns) and returns dict
def single_dataset_quality_report(df: pd.DataFrame, key_columns=None) -> dict:
    report = {
        'row_count': df.shape[0],   # Number of rows in dataframe
        'column_count': df.shape[1],# Number of columns in dataframe
        'missing_values': df.isnull().sum().to_dict(), # count of null values
        'duplicate_rows': df.duplicated().sum(),# count of duplicate rows
        'data_types': df.dtypes.apply(lambda x: x.name).to_dict(),
        'unique_values_per_column': df.nunique().to_dict()
    }

    # Anomaly check: negative values
    anomalies = {}   # empty dict to keep track of negative value count
    for col in df.columns:  # for iteration through every column name in dataframe
    #check if the column data type is numeric
        if pd.api.types.is_numeric_dtype(df[col]):
          #counts how many values are less than zero
            neg_vals = (df[col] < 0).sum()
            if neg_vals > 0:
                anomalies[col] = int(neg_vals)
    report['anomalies_negative_values'] = anomalies



    # Key columns checks if provided
    if key_columns:
      # Calculate how many duplicate rows exist when only considering those columns (subset=key_columns)
        duplicates_on_key = df.duplicated(subset=key_columns).sum()
      # Calculate the percentage of those duplicates relative to the total number of rows.
        duplicates_on_key_pct = (duplicates_on_key / len(df) * 100) if len(df) > 0 else 0
        report['duplicates_on_key_columns'] = duplicates_on_key
        report['duplicates_on_key_columns_pct'] = duplicates_on_key_pct

        # for iteration:
        for col in key_columns:
          # if column exists in Dataframe
            if col in df.columns:
              # if yes add a bolean to the report dictionary indicating values in column are unique
                report[f'unique_in_{col}'] = df[col].is_unique
            else:
              # if does not exist , add the key with value of none indicating it not found
                report[f'unique_in_{col}'] = None  # Column not found

    return report


**for two dataset case**
- **data_quality_and_compare**

- **Generate Individual Quality Reports**:
 - Creates summary reports for each input DataFrame, including row/column counts, missing values per column, duplicate rows, and data types.

- **Compare Basic Dataset Properties**:
 - Checks if row and column counts match between the two DataFrames, identifies common columns, and finds columns unique to each dataset.

- **Data Type and Missing Value Differences**:
- For columns common to both DataFrames, detects mismatches in data types and differences in counts of missing values.

- **Duplicate Row Comparison**:
- Calculates the absolute difference in the number of duplicate rows between the two DataFrames.

- **Anomaly Detection**:
- Checks for negative numeric values in the common columns of both datasets and reports counts where anomalies exist.

This function provides a comprehensive comparison and data quality assessment of two datasets, with optional key-column-based checks for deeper insights.


In [ ]:
import pandas as pd
import numpy as np

def data_quality_and_compare(df1: pd.DataFrame, df2: pd.DataFrame, key_columns=None) -> dict:
    """
    Takes two dataframes, returns data quality reports for each,
    plus a comparison report between them.

    key_columns: List of columns to use as keys for more detailed comparison (optional).
    """
    def quality_report(df):
        report = {}
        report['row_count'] = df.shape[0] # NUmber of Rows
        report['column_count'] = df.shape[1] # Number of columns
        report['missing_values'] = df.isnull().sum().to_dict() # Number of null values
        report['duplicate_rows'] = df.duplicated().sum()  # NUmber of duplicate rows.
        report['data_types'] = df.dtypes.apply(lambda x: x.name).to_dict() #
        return report

    # Individual reports
    report1 = quality_report(df1)
    report2 = quality_report(df2)

    # Comparison report between two dataframes

    compare = {}
    # check if rows and column count match between two dataframe
    compare['row_count_match'] = (report1['row_count'] == report2['row_count'])
    compare['column_count_match'] = (report1['column_count'] == report2['column_count'])
    # finds column common to both dataframe , find columns present in df1 but missing in df2 , vica versa
    compare['common_columns'] = list(set(df1.columns).intersection(set(df2.columns)))
    compare['columns_in_df1_not_in_df2'] = list(set(df1.columns) - set(df2.columns))
    compare['columns_in_df2_not_in_df1'] = list(set(df2.columns) - set(df1.columns))

    # Data type mismatch check for common columns

    dtype_mismatches = {}
    # For each common column, checks if the data types differ
    # between the two DataFrames.
    for col in compare['common_columns']:
      # If they differ, records the data types from each DataFrame in a dictionary.
        if report1['data_types'].get(col) != report2['data_types'].get(col):
            dtype_mismatches[col] = (report1['data_types'].get(col), report2['data_types'].get(col))
    compare['dtype_mismatches'] = dtype_mismatches # adds to dict to compare

    # Missing values comparison on common columns
    missing_diff = {}
    # Compares counts of missing values for each common column between
    # the two DataFrames.
    for col in compare['common_columns']:
        mv1 = report1['missing_values'].get(col, 0)
        mv2 = report2['missing_values'].get(col, 0)
        if mv1 != mv2:
            missing_diff[col] = (mv1, mv2)
    # Records differences in a dictionary, showing missing counts for
    #each DataFrame per column.
    compare['missing_value_diff'] = missing_diff

    # Duplicate rows comparison between two Dataframe
    compare['duplicate_rows_diff'] = abs(report1['duplicate_rows'] - report2['duplicate_rows'])


    # 1. Check duplicates on key columns (if provided)
    # If key_columns provided:
    if key_columns:
      # Counts duplicates in each DataFrame considering only those columns.
        compare['duplicates_on_key_columns_df1'] = df1.duplicated(subset=key_columns).sum()
        compare['duplicates_on_key_columns_df2'] = df2.duplicated(subset=key_columns).sum()
      #  Calculates what percentage these duplicates are of total rows
        compare['duplicates_on_key_columns_pct_df1'] = (compare['duplicates_on_key_columns_df1'] / len(df1) * 100) if len(df1) > 0 else 0
        compare['duplicates_on_key_columns_pct_df2'] = (compare['duplicates_on_key_columns_df2'] / len(df2) * 100) if len(df2) > 0 else 0
    else:
      # If no key columns, sets those metrics to None.
        compare['duplicates_on_key_columns_df1'] = None
        compare['duplicates_on_key_columns_df2'] = None
        compare['duplicates_on_key_columns_pct_df1'] = None
        compare['duplicates_on_key_columns_pct_df2'] = None

    # 2. Check unique constraints on key columns
    if key_columns:

      # For each key column, checks if the values in that
      # column are unique in each DataFrame.
        for col in key_columns:
            compare[f'unique_in_df1_{col}'] = df1[col].is_unique
            compare[f'unique_in_df2_{col}'] = df2[col].is_unique



    # 3. Reconciliation on key columns: rows missing in each dataframe
    if key_columns:
      #Performs an outer merge on the key columns to find rows unique to each DataFrame
        merged = pd.merge(df1, df2, on=key_columns, how='outer', indicator=True)
        # Rows in df2 but not df1 are stored in 'rows_missing_in_df1'.
        compare['rows_missing_in_df1'] = merged[merged['_merge'] == 'right_only']
        # #Rows in df1 but not df2 are stored in 'rows_missing_in_df2'
        compare['rows_missing_in_df2'] = merged[merged['_merge'] == 'left_only']
    else:
        compare['rows_missing_in_df1'] = None
        compare['rows_missing_in_df2'] = None

    # 4. Error/Anomaly detection example: negative numeric value
    anomalies = {}
    for col in compare['common_columns']:
        if pd.api.types.is_numeric_dtype(df1[col]) and pd.api.types.is_numeric_dtype(df2[col]):
            neg_df1 = (df1[col] < 0).sum()
            neg_df2 = (df2[col] < 0).sum()
            if neg_df1 > 0 or neg_df2 > 0:
                anomalies[col] = {'negative_values_in_df1': int(neg_df1), 'negative_values_in_df2': int(neg_df2)}
    compare['anomalies'] = anomalies



    return {
        'report_df1': report1,
        'report_df2': report2,
        'comparison': compare
    }


- **ai_agent_data_quality**:

- **Wrapper Function**:
- Acts as a single entry point to run data quality checks on either one or two datasets.

- **Single Dataset Mode**:
- If only one DataFrame (df1) is provided, it runs a quality report on that dataset using the single_dataset_quality_report function.

- **Two Dataset Mode**:
- If two DataFrames (df1 and df2) are provided, it runs quality reports on both and compares them using the data_quality_and_compare function.

- **Key Columns Usage**:
Supports an optional key_columns parameter to perform more detailed checks like uniqueness and reconciliation based on those columns.

- **Flexible Output Structure**:
- Always returns a dictionary with three keys:

- `report_df1`: quality report for df1

- `report_df2`: quality report for df2 or None if single dataset

- `comparison`: comparison report if two datasets, else None

- **Simplifies Calling**:
- Provides a clean and unified interface for our AI agent or other parts of our pipeline to easily get data quality insights, whether for single or paired datasets.

This function neatly orchestrates our existing quality reporting functions based on input


In [ ]:
def ai_agent_data_quality(df1: pd.DataFrame, df2: pd.DataFrame = None, key_columns=None) -> dict:
    """
    Wrapper to run data quality checks:
    - If only df1 provided: returns single dataset quality report.
    - If df1 and df2 provided: returns reports for both and their comparison.
    - key_columns used for uniqueness and reconciliation checks.

    Returns a dict with keys:
    - 'report_df1' : data quality for df1
    - 'report_df2' : data quality for df2 (if df2 given, else None)
    - 'comparison' : comparison dict (if df2 given, else None)
    """
    if df2 is None:
        # Single dataset quality report only
        report1 = single_dataset_quality_report(df1, key_columns)
        return {
            'report_df1': report1,
            'report_df2': None,
            'comparison': None
        }
    else:
        # Two dataset quality and comparison report
        combined_report = data_quality_and_compare(df1, df2, key_columns)
        return combined_report


## Step 2:NLP Summary Generator

In [ ]:
!pip install --quiet langchain langchain-community openai


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [ ]:
from getpass import getpass
import os

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")


Enter your OpenAI API key: ··········


# Design Concept
- Analyze the quality of one or two dataset case
- Data gathering and computation (counts,duplicates,nulls),
- Table generation
- prompt construction and LLM invocation,
- printing structured output

- `**functions**`
`generate_single_dataset_tables` and `generate_two_dataset_tables`.

- These are hooks for future extension. Right now they return dummy tables, but you can replace them with logic that turns the metadata into meaningful DataFrames.

- `verbose mode`:Allows the function to both:

- Return programmatic output (tables + summary), and

- Print a human-friendly view to console for debugging or display.


- `Prompt engineering`:

- You build a clear instruction + content + cue (“Summary:”) so the LLM does what you expect.

- Using PromptTemplate and LLMChain from LangChain abstracts away boilerplate of building prompts and calling models.

- `Robustness checks`:

- validate_dq_report

- Checks for missing original DataFrames

- Safeguards around dividing by zero (in pct_diff)

- Default values via .get(key, default) to avoid KeyErrors




In [ ]:
from langchain.chat_models import ChatOpenAI
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
import pandas as pd
from typing import Dict, Union


# Finding Duplicate values (Row-wise and Cell-wise)
#Identifes cell-level duplicates , but only in duplicated cell
def find_duplicate_values_table(df: pd.DataFrame) -> pd.DataFrame:
# get duplicated row (all occurances):
    duplicated_indices = df.index[df.duplicated(keep=False)].tolist()
    # returns empty result if no duplicates found
    if not duplicated_indices:
        return pd.DataFrame(columns=["row", "column_name", "duplicate_value"])

    records = []
    dup_df = df.loc[duplicated_indices]
# Iterate through each row and column in the duplicated subset:
    for idx, row in dup_df.iterrows(): # Loop through each cell of duplicated rows.
    # if value occurs more than once in its column(within duplicates)
    # records if its a true duplicate(i,e , count >1)
        for col in dup_df.columns:
            val = row[col]
            count = dup_df[col].value_counts().get(val, 0)
            if count > 1:
                records.append({
                    "row": idx,
                    "column_name": col,
                    "duplicate_value": val
                })
    # combine all records into a result Dataframe with row,column_name, DUplicate_value
    result_df = pd.DataFrame(records)
    return result_df.reset_index(drop=True)


# TO display Duplicate values Horizontally
def duplicates_horizontal_format(duplicates_df: pd.DataFrame) -> pd.DataFrame:
  # Pivot the duplicate values so each row shows duplicated values across columns (better readability).
    if duplicates_df.empty:
        return pd.DataFrame()

    pivot_df = duplicates_df.pivot(index='row', columns='column_name', values='duplicate_value')
    pivot_df = pivot_df.reset_index()
    pivot_df = pivot_df.sort_values(by='row').reset_index(drop=True)

    return pivot_df

# Summmarizing Null values
# Count null values per column
def summarize_nulls(df: pd.DataFrame) -> pd.DataFrame:
  # df.isnull(): create a boolean mask of shape as df
  # every cell become True if value is null else false otherwise
    null_counts = df.isnull().sum()
  #null_count> 0 : filter out columns with zero null
  #null_count[null_counts>0]: Applies the filter to keep only relevant columns
  #sort_values(ascending = False): sorts the resulting series in desceding order so that column with most missing values appear at top
    null_counts = null_counts[null_counts > 0].sort_values(ascending=False)
# Checks if there are no columns with nulls.
    if null_counts.empty:
        return pd.DataFrame(columns=['column_name', 'null_count'])
# null_count is currently a pandas series .rest_index() turns into dataframe
    summary_df = null_counts.reset_index()
    summary_df.columns = ['column_name', 'null_count']
    return summary_df

# Combined Dataset Cleanliness Check
def analyze_dataset_clean(df: pd.DataFrame):
  # finds duplicate_value_table, horizantal_format, summarize_null
    duplicates = find_duplicate_values_table(df)
    duplicates_formatted = duplicates_horizontal_format(duplicates)
    nulls_summary = summarize_nulls(df)
# bundles their output into a single dictionary:
    return {
        "duplicates": duplicates_formatted,
        "nulls_summary": nulls_summary
    }

# to validate the input for data quality report. it enusres that `dq_report` passes into other functions is a python dictionary
def validate_dq_report(dq_report): # dq_report expected input - usally a dictionary that contains:one or two dataset reports
    if not isinstance(dq_report, dict):
        raise ValueError("dq_report must be a dictionary.")

#This is a placeholder function that returns dummy table(s). It’s meant to eventually generate useful tables from a single dataset’s report
def generate_single_dataset_tables(rep):

    return {"example_table": pd.DataFrame({"col1": [1, 2], "col2": [3, 4]})}


def generate_two_dataset_tables(r1, r2, c):
   # This is a placeholder function that returns dummy table(s). It’s meant to eventually generate useful tables from a single dataset’s report
    return {
        "table1": pd.DataFrame({"A": [1, 2]}),
        "table2": pd.DataFrame({"B": [3, 4]})
    }


def generate_and_visualize_dq_report(
    dq_report: dict, # dictionary containing the report metadata and raw Dataframe
    model_name: str = "gpt-4o-mini",
    temperature: float = 0.0, # tunning or controlling randomness of GPT output
    verbose: bool = True # whether to print summaries to console
) -> Dict[str, Union[Dict[str, pd.DataFrame], str]]:
# calling helper function which checks dq_report is a dictionary, if not it raises a value error
    validate_dq_report(dq_report)

    results = {} # empty list dict that will collect all tabular output
    summary_info = "" # is a temp string used to build a textual  summmary of key metrics
# this checks if dictionary has a second report or not or lacks a comparsion strucutre, then we treat it as a single dataset case
    single_dataset_case = dq_report.get('report_df2') is None or dq_report.get('comparison') is None

# case A: A single dataset
    if single_dataset_case:
      # rep is the summary data dict for dataset 1(with counts , missing values, duplicates,etc)
        rep = dq_report['report_df1']
        #df1 is actual pandas Dataframe(raw data)
        df1 = dq_report.get('df1_original')
        # if raw dataframe isn't present in the input , that's an error
        if df1 is None:
            raise ValueError("Original dataframe df1 must be included in dq_report as 'df1_original'.")
       # extracting basic statistics:
        row_count = rep['row_count']
        col_count = rep['column_count']
        duplicate_count = rep['duplicate_rows']
        null_count = sum(rep.get('missing_values', {}).values()) # rep.get('missing_values',{}) ensure if key is missing, you get an empty dict(sum=0)
# simple summary string:
        summary_info = (
            # fstring organizes the key metrics in human readble block
            f"Data Quality Check Report:\n"
            f"----------------------------\n"
            f"Row Count (df1): {row_count}\n"
            f"Column Count (df1): {col_count}\n"
            f"Duplicate Rows Count: {duplicate_count}\n"
            f"Null Values Count: {null_count}\n"
        )
        # it will be printed (if verbose=True) to show a quick snapshot of dataset
# calling helper fn generate_single_dataset_tables with rep right now we are not passing any dataframe in this so we get dummy dataframe
# if we pass dataframe we will get the result
        results = generate_single_dataset_tables(rep)
# compute detailed duplicates/null summaries using actual df1
        analysis = analyze_dataset_clean(df1) # this function will return dataframe of duplicate detials , summary of null count
        # then insert these into results under keys 'duplicates_detail' and 'nulls_summary'.
        results['duplicates_detail'] = analysis['duplicates']
        results['nulls_summary'] = analysis['nulls_summary']

        if verbose: # if verbose is true it prints the summmary and details
            print(summary_info)
            print("\nDuplicate Detail:")
            print("-----------------")
            if results['duplicates_detail'].empty:
                print("No duplicate values found.")
            else:
                print(results['duplicates_detail'].to_string(index=False))

            print("\nNull Summary:")
            print("-------------")
            if results['nulls_summary'].empty:
                print("No null values found.")
            else:
                print(results['nulls_summary'].to_string(index=False))

# Case B : Two dataset (case)
    else:
      # r1, r2 metadata dict for dataset1 and dataset2
        r1 = dq_report['report_df1']
        r2 = dq_report['report_df2']
        c = dq_report['comparison'] # c: comparision metadata(common columns , mismatches, differences)
        # df1,df2 : raw dataframe for each set
        df1 = dq_report.get('df1_original')
        df2 = dq_report.get('df2_original')

       # if df1 and df2 dataframe is missing, error out
        if df1 is None or df2 is None:
            raise ValueError("Original dataframes df1 and df2 must be included in dq_report as 'df1_original' and 'df2_original'.")
      # extract basic stats from both:
        row_count_1 = r1['row_count']
        row_count_2 = r2['row_count']
        col_count_1 = r1['column_count']
        col_count_2 = r2['column_count']

        dup_count_1 = r1['duplicate_rows']
        dup_count_2 = r2['duplicate_rows']

        null_count_1 = sum(r1.get('missing_values', {}).values())
        null_count_2 = sum(r2.get('missing_values', {}).values())

    # Extract comparision metadata: (data comparision between two dataset)

        common_cols = c.get('common_columns', []) # column which are common between those two dataset
        cols_only_df1 = c.get('columns_in_df1_not_in_df2', []) # unique column
        cols_only_df2 = c.get('columns_in_df2_not_in_df1', []) # unique column
        dtype_mismatches = c.get('dtype_mismatches', {})
        rows_only_df1 = c.get('rows_only_in_df1', []) # rows appearing only in one dataset
        rows_only_df2 = c.get('rows_only_in_df2', [])
        identical_rows = c.get('identical_rows', []) # identical rows across dataset
    # total number of values (cells) in each dataset: rows × columns
        total_values_df1 = row_count_1 * col_count_1
        total_values_df2 = row_count_2 * col_count_2
    # small helper function to compute percentage difference:
        def pct_diff(a, b):
          # (abs(a - b) / ((a + b) / 2)) * 100 is a formula for percentage difference relative to the average.
          #If a + b is zero (i.e., both are zero), return 0 (to avoid division by zero).
            return abs(a - b) / ((a + b) / 2) * 100 if (a + b) > 0 else 0
# compute difference between dataset and matches:
# pct_difference: how far apart dataset are in terms of total "cells"
        pct_difference = pct_diff(total_values_df1, total_values_df2)
        # total_values_matches : number of cell match . if there are no common columns its zeri
        total_values_matches = len(identical_rows) * len(common_cols) if common_cols else 0

        summary_info = (
            f"Data Quality Check Report:\n"
            f"----------------------------\n"
            f"Row Count (df1): {row_count_1}\n"
            f"Row Count (df2): {row_count_2}\n"
            f"Column Count (df1): {col_count_1}\n"
            f"Column Count (df2): {col_count_2}\n"
            f"Duplicate Rows Count (df1): {dup_count_1}\n"
            f"Duplicate Rows Count (df2): {dup_count_2}\n"
            f"Null Values Count (df1): {null_count_1}\n"
            f"Null Values Count (df2): {null_count_2}\n"
            f"Common Columns: {common_cols}\n"
            f"Unique Columns in df1: {cols_only_df1}\n"
            f"Unique Columns in df2: {cols_only_df2}\n"
            f"Schema Mismatches: {dtype_mismatches}\n"
           # f"Rows Only in Dataset 1 Count: {len(rows_only_df1)}\n"
            f"Rows Only in Dataset 2 Count: {len(rows_only_df2)}\n"
            f"Identical Rows in Both Datasets Count: {len(identical_rows)}\n"
            f"Total Values Matches Count: {total_values_matches}\n"
            f"Percentage Difference between datasets: {pct_difference:.2f}%\n"
        )

        results = generate_two_dataset_tables(r1, r2, c)
      # Compute quality analyses for both datasets
  # Uses your earlier analyze_dataset_clean to get duplicate and null summaries for both datasets
  #and inserts them into results.
        analysis_df1 = analyze_dataset_clean(df1)
        analysis_df2 = analyze_dataset_clean(df2)

        results['duplicates_detail_df1'] = analysis_df1['duplicates']
        results['nulls_summary_df1'] = analysis_df1['nulls_summary']
        results['duplicates_detail_df2'] = analysis_df2['duplicates']
        results['nulls_summary_df2'] = analysis_df2['nulls_summary']

        if verbose:
            print(summary_info)

            print("\nDuplicate Detail - Dataset 1:")
            print("----------------------------")
            if results['duplicates_detail_df1'].empty:
                print("No duplicate values found in Dataset 1.")
            else:
                print(results['duplicates_detail_df1'].to_string(index=False))

            print("\nNull Summary - Dataset 1:")
            print("-------------------------")
            if results['nulls_summary_df1'].empty:
                print("No null values found in Dataset 1.")
            else:
                print(results['nulls_summary_df1'].to_string(index=False))

            print("\nDuplicate Detail - Dataset 2:")
            print("----------------------------")
            if results['duplicates_detail_df2'].empty:
                print("No duplicate values found in Dataset 2.")
            else:
                print(results['duplicates_detail_df2'].to_string(index=False))

            print("\nNull Summary - Dataset 2:")
            print("-------------------------")
            if results['nulls_summary_df2'].empty:
                print("No null values found in Dataset 2.")
            else:
                print(results['nulls_summary_df2'].to_string(index=False))
    # Prints the summary, duplicate tables, null summaries for both datasets (with checks for “empty”)

    #Formatting the prompt text for the LLM
    # inside this outer function , two inner helper function are defined
    # one for single dataset case  and another for two-dataset case

    # case A: single dataset case
    def format_single_report(rep):
        total_missing = sum(rep.get('missing_values', {}).values())
        total_rows = rep.get('row_count', 0)
        duplicate_rows = rep.get('duplicate_rows', 0)
        duplicate_pct = (duplicate_rows / total_rows * 100) if total_rows > 0 else 0
        lines = [
            f"Dataset has {total_rows} rows and {rep.get('column_count', 0)} columns.",
            f"Duplicate rows: {duplicate_rows} ({duplicate_pct:.2f}%)",
            f"Total missing values: {total_missing}",
            f"Missing values per column: {rep.get('missing_values', {})}",
            f"Data types per column: {rep.get('data_types', {})}",
            f"Unique values per column: {rep.get('unique_values_per_column', 'N/A')}",
            f"Locations of null values: {rep.get('null_value_locations', 'N/A')}",
            f"Duplicate rows found at indices: {rep.get('duplicate_row_indices', 'N/A')}"
        ]
        return "\n".join(lines)
       # this builds a human readble text snippet that describes key aspects of single dataset
       # number of rows/columns, duplicates, missing values, metadata like data_types, unique values, etc. (assuming rep contains those keys)
       #It returns a multi-line string ("\n".join(lines)) that is used in the prompt to GPT.


  # Case B: two dataset

    def format_two_dataset_report(report):
        r1 = report['report_df1']
        r2 = report['report_df2']
        c = report['comparison']

        total_missing_1 = sum(r1.get('missing_values', {}).values())
        total_missing_2 = sum(r2.get('missing_values', {}).values())
        duplicate_rows_1 = r1.get('duplicate_rows', 0)
        duplicate_rows_2 = r2.get('duplicate_rows', 0)
        duplicate_pct_1 = (duplicate_rows_1 / r1.get('row_count', 1) * 100) if r1.get('row_count', 0) > 0 else 0
        duplicate_pct_2 = (duplicate_rows_2 / r2.get('row_count', 1) * 100) if r2.get('row_count', 0) > 0 else 0

        lines = [
            f"Dataset 1 has {r1.get('row_count', 0)} rows and {r1.get('column_count', 0)} columns.",
            f"Dataset 2 has {r2.get('row_count', 0)} rows and {r2.get('column_count', 0)} columns.",
            f"Duplicate rows: Dataset 1 - {duplicate_rows_1} ({duplicate_pct_1:.2f}%), Dataset 2 - {duplicate_rows_2} ({duplicate_pct_2:.2f}%)",
            f"Total missing values: Dataset 1 - {total_missing_1}, Dataset 2 - {total_missing_2}",
            f"Common columns: {c.get('common_columns', [])}",
            f"Columns only in Dataset 1: {c.get('columns_in_df1_not_in_df2', [])}",
            f"Columns only in Dataset 2: {c.get('columns_in_df2_not_in_df1', [])}",
            f"Schema mismatches (data types): {c.get('dtype_mismatches', {})}",
          #  f"Rows only in Dataset 1: {c.get('rows_only_in_df1', [])}",
           # f"Rows only in Dataset 2: {c.get('rows_only_in_df2', [])}",
            f"Identical rows in both datasets: {c.get('identical_rows', [])}"
        ]
        return "\n".join(lines)
# Similar to format_single_report, but includes comparative metrics and difference-related metadata.
# then our code picks which format to use and build the prompt to send to GPT:
    if single_dataset_case:
      # report text : become the structured content generated above.
      #prompt text: is an instruction + the report , ending summary which uses GPT to produce a summarization
        report_text = format_single_report(dq_report['report_df1'])
        prompt_text = (
            "You are a data quality expert. "
            "Given the following data quality report for a single dataset, "
            "generate a concise and insightful human-readable summary highlighting key findings and issues.\n\n"
            f"Report:\n{report_text}\n\nSummary:"
        )
    else:
        report_text = format_two_dataset_report(dq_report)
        prompt_text = (
            "You are a data quality expert. "
            "Given the following data quality and comparison report between two datasets, "
            "generate a concise and insightful human-readable summary highlighting key findings, issues, and differences.\n\n"
            f"Report:\n{report_text}\n\nSummary:"
        )
# calling LLLM via Lang-Chain
# creates a prompt_template that excepts a single variable named "report"
    prompt_template = PromptTemplate(
        input_variables=["report"],
        template="{report}"
    )
    llm = ChatOpenAI(temperature=temperature, model=model_name)
    chain = LLMChain(llm=llm, prompt=prompt_template) # LLMChain combines the model and prompt template.
    summary = chain.run(report=prompt_text) # calls chain.run(report=prompt_text)
# Populates the prompt template with prompt_text,Sends it to the OpenAI model,
#Receives the generated summary (string).
    if verbose:
        print("\nNLP Summary:")
        print("------------")
        print(summary)

    return {
        'tables': results,
        'summary': summary
    }


In [ ]:
from google.colab import files
import io
import pandas as pd

# defining a wrapper utility function to:
# it will allow user to upload 1 or 2 CSV files
# Run a Data quality agent function  on them
# generate and display result

def upload_and_run_dq_report(ai_agent_data_quality_func, generate_and_visualize_dq_report_func):
    """
    Upload 1 or 2 CSV files (depending on your DQ check),
    run the AI data quality agent on them,
    then visualize and print the summary and tables.

    Args:
        ai_agent_data_quality_func (callable): Function to generate DQ report dict from dataframe(s).
        generate_and_visualize_dq_report_func (callable): Your existing function that takes dq_report dict
                                                          and prints tables, summary, and visuals.
    """
    print("Please upload 1 or 2 CSV files for data quality check...")
    uploaded = files.upload()
# it checks user uploaded 1 or 2 files if not prints an error
    if len(uploaded) not in [1, 2]:
        print("Error: Please upload exactly 1 or 2 CSV files.")
        return

    file_names = list(uploaded.keys())
    dfs = []
    for f in file_names:
        dfs.append(pd.read_csv(io.BytesIO(uploaded[f]))) # io.BytesIO()-converts file byte into file-like object

    print(f"Loaded files: {file_names}")

    # Run AI agent data quality check based on number of files uploaded
    # for single file:
    # calls ai_agent_data_quality_func(df1)
    if len(dfs) == 1:
        dq_report = ai_agent_data_quality_func(dfs[0])
        # Stores the original DataFrame back into the DQ report under 'df1_original' → so it can later be analyzed or visualized
        dq_report['df1_original'] = dfs[0]
  # for two files:
  # calls ai_agent_data_quality_func(df1,df2)
    else:
        dq_report = ai_agent_data_quality_func(dfs[0], dfs[1])
        # stores both df into report dict
        dq_report['df1_original'] = dfs[0]
        dq_report['df2_original'] = dfs[1]

    print("Generating and visualizing data quality report...\n")
    results = generate_and_visualize_dq_report_func(dq_report)
    return results


In [ ]:
results = upload_and_run_dq_report(ai_agent_data_quality, generate_and_visualize_dq_report)


Please upload 1 or 2 CSV files for data quality check...


Saving sample_dataset_2.csv to sample_dataset_2 (2).csv
Saving sample_dataset.csv to sample_dataset (1).csv
Loaded files: ['sample_dataset_2 (2).csv', 'sample_dataset (1).csv']
Generating and visualizing data quality report...

Data Quality Check Report:
----------------------------
Row Count (df1): 53
Row Count (df2): 50
Column Count (df1): 5
Column Count (df2): 8
Duplicate Rows Count (df1): 3
Duplicate Rows Count (df2): 0
Null Values Count (df1): 11
Null Values Count (df2): 13
Common Columns: ['Name', 'ID', 'Age', 'Salary']
Unique Columns in df1: ['City']
Unique Columns in df2: ['Rating', 'Department', 'BonusPct', 'JoinDate']
Schema Mismatches: {}
Rows Only in Dataset 2 Count: 0
Identical Rows in Both Datasets Count: 0
Total Values Matches Count: 0
Percentage Difference between datasets: 40.60%


Duplicate Detail - Dataset 1:
----------------------------
 row   Age    City ID   Name   Salary
   6  47.0  Boston 28 Name28  64538.0
  17  58.0 Chicago 36 Name36  57412.0
  32  58.0 Chicag